# STAT 450: Case Studies in Statistics

#  Case study: Relation between mRNA and protein levels 

### Recall the client's claim

Using the median ratio of protein to mRNA levels per gene as a proxy for translation rates, our data show that [...] ***it now becomes possible to predict protein abundance in any given tissue with good accuracy from the measured mRNA abundance***


In this notebook, we continue to consider ways to explore and model how protein levels can depend on mRNA levels.

Monday class: For exploring:
-  scatterplots
-  correlations

Monday class: We looked at our client's model and estimates:

For gene $g$, tissue $t$: 
$$
{\rm{protein}}_{gt}  \approx  \beta_g \times ({\rm{ mRNA }}_{gt}) $$

The client uses:
$$\hat{\beta}_g  = {\rm{  median ~of~ the ~12 ~ratios~of~}} 
\frac{{\rm{protein}}_{gt}}{ {\rm{ mRNA }}_{gt}} $$

<h4 style="color:red; font-weight:bold;">Discussion:</h4>    Why might the client think that intercept=0 is reasonable?  Think of the biology. (Hint: How is protein made?) 

**Today**: For other types of models: we will consider least squares linear regression.  All will have as response a continuous real-valued y (protein).  For covariates, we'll consider:
-  one continuous (real-valued) covariate  (mRNA level)
   -  with and without intercept
   - with interaction  (parameters can depend on gene, so gene is a covariate)
   
**Today**:  lots of new R code!  `lm`, `nest`,`map`,`abline`, `anova`

**Today**: an F test to compare nested models via ANOVA `anova`
   

-----------------------

### Load libraries and read in the tidy data file data/tidy_data  

Note that you now have a data set `data/tidy_data`, so that you no longer need to wrangle the two original data files to get the tibble `data_sel`.

In [ ]:
# load libraries
library(tidyverse)
library(broom)
dat_sel <-  read_csv("data/tidy_data.csv",show_col_types = FALSE)
head(dat_sel)

-----------------------

 Choose 2 genes at random from those genes with **12 complete pairs** using `filter`. 

In [ ]:
set.seed(450)
dat_2genes <- dat_sel %>% 
              filter(values.available==12) %>%
              filter(gene %in% sample(unique(gene), 2))
   
dat_2genes

Make 2 scatterplots, one for each gene.

In [ ]:
dat_2genes %>% ggplot(aes(x=mrna,y=protein)) +
              geom_point() +
              facet_wrap(.~gene) +
        xlab("mRNA") + ylab("Protein")  +
    ggtitle("Scatterplot of Protein versus mRNA levels of 2 Randomly Chosen Genes") +
    xlim(0,NA)

-----------------------

## Use usual linear models, fit via least squares

For each fixed gene $g=1,2$, for each tissue $t=1,\ldots, 12$  (should give tissue names ....),  here are 4 models:

$$
{\rm{A. ~intercept=0}}:\quad 
{\rm{protein}}_{gt} =  \beta_g \times ({\rm{ mRNA }}_{gt}) + \epsilon_{gt}, 
\quad \epsilon_{gt} \sim N(0,\sigma_{g}^2)
$$


$$
{\rm{B. ~intercept=0}}: \quad 
{\rm{protein}}_{gt} =  \beta_g \times ({\rm{ mRNA }}_{gt}) + \epsilon_{gt}, 
\quad \epsilon_{gt} \sim N(0,\sigma^2)
$$

<h4 style="color:red; font-weight:bold;">Discussion:</h4> What is the difference between models A and B?  
 <p>&nbsp;</p>    
   
      
      
   
        



$$
{\rm{C.}}\quad 
{\rm{protein}}_{gt} = \alpha_g + \beta_g \times ({\rm{ mRNA }}_{gt}) + \epsilon_{gt}, 
\quad \epsilon_{gt} \sim N(0,\sigma_g^2)
$$

$$
{\rm{D.}}\quad 
{\rm{protein}}_{gt} = \alpha_g + \beta_g \times ({\rm{ mRNA }}_{gt}) + \epsilon_{gt},
\quad \epsilon_{gt} \sim N(0,\sigma^2)
$$

<h4 style="color:red; font-weight:bold;">Discussion:</h4> 

- What is the difference between models C and D?

- What is the difference between models B and D?

- From the two scatterplots above, what do you think of models A and B compared to models C and D?


Same variances:  
- fit one big model $Y=X \beta + \epsilon$ using one `lm` 

Different variances: 
- fit separate models for each gene: 
  -  low tech: split data set in two and analyze each separately
  -   fancy:  (better if more than 2 genes) use gene-by-gene `lm` by using `nest` and `map`
- can plot via `facet_wrap`

Comment:  for the models considered here it ends up that 
- models A and B have the same slope estimates
   -  model A has two estimates of variance
   -  model B has one estimate of variance
- models C and D have the same slopes and intercept estimates

## Consider Models B and D first

### Model B
**intercept=0, same error variance for all genes**
$$
{\rm{protein}}_{gt} = \beta_g \times  ({\rm{ mRNA }}_{gt}) + \epsilon_{gt}, \quad \epsilon_{gt} \sim N(0,\sigma^2)
$$

Same variances, so this is the usual $Y=X \beta + \epsilon$ model:  so fit with one `lm`.


In [ ]:
lm_B <- lm(data=dat_2genes,protein ~ 0 + gene:mrna)
lm_B
names(lm_B)

In [ ]:
coef(lm_B)

In [ ]:
model.matrix(lm_B)

In [ ]:
summary(lm_B)

#### PLOT MODEL B


Fancy trick!  Use the fitted values from `lm_B` to make a line with `geom_line`.

In [ ]:
fitted(lm_B)

In [ ]:
dat_Bfitted <-  dat_2genes %>%
    mutate(yhat=fitted(lm_B))
dat_Bfitted

In [ ]:
dat_Bfitted %>% 
    ggplot(aes(x=mrna,y=yhat)) +
    facet_wrap(.~gene) +
    geom_point()

In [ ]:
dat_Bfitted %>%  
ggplot(aes(x=mrna,y=protein)) +
   geom_point() +
   facet_wrap(.~gene) +
   geom_line(aes(x=mrna,y=yhat)) +
   xlab("mRNA") +
   ylab("Protein") +
   xlim(0,NA)


### Model D
**Different slope and intercept for each gene, same error variance for all genes**
$$
{\rm{protein}}_{gt} =\alpha_g+ \beta_g \times  ({\rm{ mRNA }}_{gt}) + \epsilon_{gt},
\quad \epsilon_{gt} \sim N(0,\sigma^2)
$$

Same variances, so this is the usual $Y=X \beta + \epsilon$ model:  so fit with one `lm`.


In [ ]:
lm_D <- lm(data=dat_2genes, protein ~ gene*mrna)
model.matrix(lm_D)
summary(lm_D)


<h4 style="color:red; font-weight:bold;">Discussion:</h4> 

$\hat{Y}= X \hat{\beta}$:  How do the estimated coefficients relate to the slopes and intercepts of the two lines?
Hint:  look at the model matrix.

### Plot Model D

<h4 style="color:red; font-weight:bold;">Exercise:</h4> 
Make two scatterplots and put the estimated lines on them.

Use the "yhat"s way to draw the estimated lines.

Add the fitted values from `lm_D` into `dat_D`:

In [ ]:
###  Enter your code:
dat_D <- dat_2genes %>%
    mutate(yhat=...))
dat_D

   

Now make the scatter plot and add the lines. 

In [ ]:
## Enter your code
dat_D %>%  
ggplot(aes(x=mrna,y=protein)) +
   geom_...() +
   facet_wrap(.~gene) +
   geom_...(aes(x=mrna,y=yhat)) +
   xlab("mRNA") +
   ylab("Protein")


## Now consider Models A and C

### MODEL A 
**intercept=0, allows for different error variances for each gene**
$$
{\rm{protein}}_{gt} = \beta_g \times  ({\rm{ mRNA }}_{gt}) + \epsilon_{gt},  \quad \epsilon_{gt} \sim N(0,\sigma_g^2)
$$

This requires fitting the data from the two genes separately.  This can be done via
- `nest` and `map`
-  perhaps `group_by(gene)` but this isn't elegant

Fancy code ahead!  in steps.  

First, recall:

In [ ]:
head(dat_2genes)

`nest` takes named columns and makes them a list.

Next:  take named columns (tissue, mrna, protein) and put them in `data.list`

In [ ]:
dat_2genes %>%
    nest(data.list = c(tissue, mrna, protein)) 
     

Next:  use `map` with that `data.list`:

`map`:  lets you iterate

  map(what you iterate over, the function you apply)

In [ ]:
dat_2genes %>%
    nest(data.list = c(tissue, mrna, protein)) %>% 
        #  we did this above - creates the list and passes the new tibble along to:
    mutate(model = map(data.list, ~lm(protein ~ mrna-1, data = .x)))  
        #  here, map applies lm to every item in the data list (the two genes)
        #  ~lm is an anoymous function  .x = argument which we specify in first argument of map
        #  We added another column.  What does it contain? 
   

Finally:  get the model matrix and the slope.

In [ ]:
lm_A <-  dat_2genes %>%
    nest(data.list = c(tissue, mrna, protein)) %>% 
        #  creates the list and passes the new tibble along to:
    mutate(model = map(data.list, ~lm(protein ~ mrna-1, data = .x)), 
# did up to here in last code cell, will add:
           model_summary = map(model, ~summary(.x)),
           model_matrix = map(model,~model.matrix(.x)),
           slope = map_dbl(model,~coef(.x)))   
lm_A

Let's look at the model matrix and analysis output and use them to write the model matrix and the estimated coefficients.

In [ ]:
lm_A$model_matrix
lm_A$model_summary



##### PLOT  MODEL A
Plot the resulting linear models.  Note that `facet_wrap` makes a different model fitting for each gene, which is what we did above.

In [ ]:
ggplot(dat_2genes, aes(x = mrna, y = protein)) + 
    geom_point() + 
    facet_wrap(.~gene) + 
    geom_smooth(method = "lm", se=FALSE, color="black", formula = y ~ x, fullrange = T) + 
    xlim(0,NA)

### MODEL C
<h4 style="color:red; font-weight:bold;">(for you to look at on your own)</h4> 

***Different slope and intercept and error variance for each gene***

$${\rm{protein}}_{gt} = \alpha_g + \beta_g \times ({\rm{ mRNA }}_{gt}) + \epsilon_{gt} 
\quad \epsilon_{gt} \sim N(0,\sigma_{gt}^2)$$

In [ ]:
lm_C <-  dat_2genes %>%
    nest(data.list = c(tissue, mrna, protein)) %>% 
        #  creates the list and passes the new tibble along to:
    mutate(model = map(data.list, ~lm(protein ~ mrna, data = .x)), 
           model_summary = map(model, ~summary(.x)),
           model_matrix = map(model,~model.matrix(.x)),
           yhat = map(model,~fitted(.x))  )  
lm_C

In [ ]:
lm_C$model_summary


**Note that intercept for the first gene is significantly different from zero.**

**Note that the slope for the first gene is (possibly) negative!  but not significantly different from 0. 

Plot the results, scatterplots plus lines, with the trick of using fitted values and `geom_abline`.

In [ ]:
lm_Cfitted <- lm_C %>% select(gene, data.list,yhat)
lm_Cfitted
lm_Cfitted <- unnest(lm_Cfitted,c(data.list,yhat))  ## I can't unnest with lm objects in there!
lm_Cfitted

In [ ]:
ggplot(lm_Cfitted, aes(x = mrna, y = protein)) + 
    geom_point() + 
    facet_wrap(.~gene) + 
    geom_line(aes(x=mrna,y=yhat),lwd=1) +
    xlab("mRNA") + ylab("Protein") +
    xlim(0,NA)

## Plotting model B with `nest` and `abline` <h4 style="color:red; font-weight:bold;">(for you to look at on your own)</h4> 

Find the slope from `lm_B` and use `geom_line`  (risky because `lm_B` might have changed order of genes from `dat_2genes`)

Step 1:

In [ ]:
dat_B <- dat_2genes %>%
    nest(data = c(tissue, mrna, protein)) %>% 
    mutate(slope = coef(lm_B)) %>%
    unnest(c(data)) 

dat_B

Now plot with `geom_line`

In [ ]:
dat_B %>%    ggplot(aes(x = mrna, y = protein,color=gene)) + 
    geom_point() +
    geom_abline(aes(intercept = 0, slope = slope,color=gene), lwd=1) + 
    xlim(0,NA) +
    xlab("mRNA") + ylab("Protein")
    

### NESTED MODELS B AND D, TESTING AND F-TESTS

Used for linear model when the error variances are all $\sigma^2$ ($Y=X \beta + \epsilon$)

Consider the two models B and D.

Test:  H$_0$:  Model B holds  versus $H_a$:  Model D holds


Re-write in standard Stat 306 form and re-express the hypotheses in terms of model parameters.

Let 
- $Y_1,\ldots,Y_{12}$ be the protein values for the first gene 
-  $Y_{13},\ldots,Y_{24}$ be the protein values for the second gene 
- $I_i = 1 $ if gene 2 yielded the $i$th observation, 0 else, for $i=1,\ldots,24$. 

Then
$$
{\rm{D.}}\quad 
{\rm{protein}}_{gt} =~ \alpha_g ~+ ~\beta_g \times ({\rm{ mRNA }}_{gt}) ~+ ~\epsilon_{gt},
\quad \epsilon_{gt} \sim N(0,\sigma^2)
$$
 becomes
 $$Y_i  =  \alpha_1 ~+ ~\alpha_2 I_i  ~+~ \alpha_3 \times {\rm{ mRNA }}_i ~+ ~\alpha_4 \times {\rm{ mRNA }}_i\times I_i 
 ~ +~ \epsilon_i $$
and
$$
{\rm{B. ~intercept=0}}: \quad 
{\rm{protein}}_{gt} =  ~ \beta_g \times ({\rm{ mRNA }}_{gt})~ + ~\epsilon_{gt}, 
\quad \epsilon_{gt} \sim N(0,\sigma^2)
$$
becomes
$$  Y_i  = \alpha_3 \times {\rm{ mRNA }}_i ~+ ~\alpha_4 \times {\rm{ mRNA }}_i  \times I_i  + \epsilon_i
$$



They are "nested":  Model B is a special case of Model D:  Model B is Model D with $\alpha_1=0, \alpha_2=0$.


Test:  H$_0$:  $\alpha_1=0, \alpha_2=0$   versus H$_a$: at least one of  $\alpha_1$ or $ \alpha_2$ is non-zero.


In [ ]:
# F-test 
anova(lm_B, lm_D)


**Conclusion?**

 <h4 style="color:red; font-weight:bold;">Exercise for class if time, else at home</h4>

Consider a new model E:
$$
{\rm{E.}}\quad 
{\rm{protein}}_{gt} =~ \alpha_g ~+ ~\beta \times ({\rm{ mRNA }}_{gt}) ~+ ~\epsilon_{gt},
\quad \epsilon_{gt} \sim N(0,\sigma^2)
$$
 
<h4 style="color:red; font-weight:bold;">Exercise:</h4> Fit the model with `lm` and check with `model.matrix`.  Did `lm` fit the correct model?

In [ ]:
## Enter you code here
lm_E <- lm( data=dat_2genes,protein~ ... + ...)
model.matrix(...)

**Compare model E to model D:**   Recall
$$
{\rm{D.}}\quad 
{\rm{protein}}_{gt} =~ \alpha_g ~+ ~\beta_g \times ({\rm{ mRNA }}_{gt}) ~+ ~\epsilon_{gt},
\quad \epsilon_{gt} \sim N(0,\sigma^2)
$$
 $$Y_i  =  \alpha_1 ~+ ~\alpha_2 I_i  ~+~ \alpha_3 \times {\rm{ mRNA }}_i ~+ ~\alpha_4 \times {\rm{ mRNA }}_i\times I_i 
 ~ +~ \epsilon_i $$
 
 
<h4 style="color:red; font-weight:bold;">Exercise:</h4>
Write down model E in this form by eliminating the terms that aren't needed in the model D equation.
 $${\rm{E.}} \quad
 Y_i  =   .... \quad ...  ~ +~ \epsilon_i $$

 <h4 style="color:red; font-weight:bold;">Exercise:</h4>
 
 What are the null and alternative hypotheses for testing model E versus model D, in terms of model parameters?
 

Ho:  ???                       H_1????

 Test your null hypothesis using `anova`.  What do you conclude?

 

In [ ]:
### fill in the code here:
anova(...,...)